In [1]:
import requests
import json

# Hàm test từng trường hợp cụ thể
def debug_cafef_api_scenario(scenario_name, end_date_value):
    print(f"\n----- 🧪 TEST CASE: {scenario_name} (EndDate = '{end_date_value}') -----")
    
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/FinanceReport.ashx"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    params = {
        "Symbol": "HRC",
        "Type": 1,            # 1: Kết quả kinh doanh
        "ReportType": "QUY",  # Lấy theo Quý
        "TotalRow": 100,      # Lấy nhiều dòng
        "EndDate": end_date_value # Tham số nghi vấn
    }
    
    try:
        # 1. In ra URL thực tế để click thử
        req = requests.Request('GET', url, params=params, headers=headers)
        prepared = req.prepare()
        print(f"🔗 URL Thực tế: {prepared.url}")
        
        # 2. Gửi Request
        r = requests.Session().send(prepared)
        
        # 3. Phân tích Status Code
        print(f"📡 Status Code: {r.status_code}")
        
        if r.status_code == 200:
            data = r.json()
            
            # 4. Kiểm tra cấu trúc JSON trả về
            # API CafeF thường trả về: {"Data": {"Value": [...]}} HOẶC {"Data": null}
            
            if 'Data' in data and data['Data'] is not None:
                item_count = data['Data'].get('Count', 0)
                values = data['Data'].get('Value', [])
                print(f"✅ THÀNH CÔNG! Tìm thấy {item_count} bản ghi.")
                if len(values) > 0:
                    print(f"   Dữ liệu mới nhất: Quý {values[0].get('Quater')}/{values[0].get('Year')}")
            else:
                print("❌ THẤT BẠI: Server trả về JSON hợp lệ nhưng Data là NULL.")
                print("   -> Đây chính là nguyên nhân lỗi 'NoneType is not subscriptable'")
        else:
            print("❌ Lỗi mạng hoặc Server chặn.")
            
    except Exception as e:
        print(f"❌ Lỗi Code: {e}")

# --- CHẠY THỰC NGHIỆM ---
print("Đang chẩn đoán API Báo Cáo Tài Chính (QUÝ)...")

# Case 1: Không truyền ngày (Hy vọng server tự hiểu lấy mới nhất)
debug_cafef_api_scenario("Để trống ngày", "")

# Case 2: Truyền Năm (Code cũ của chúng ta)
debug_cafef_api_scenario("Chỉ truyền Năm", "2025")

# Case 3: Truyền dạng Quý/Năm (Giống link trình duyệt bạn gửi)
# Link trình duyệt là 'EndDate=1-2028'. Ta thử '12-2025'
debug_cafef_api_scenario("Truyền Tháng-Năm", "12-2025")

Đang chẩn đoán API Báo Cáo Tài Chính (QUÝ)...

----- 🧪 TEST CASE: Để trống ngày (EndDate = '') -----
🔗 URL Thực tế: https://cafef.vn/du-lieu/Ajax/PageNew/FinanceReport.ashx?Symbol=HRC&Type=1&ReportType=QUY&TotalRow=100&EndDate=
📡 Status Code: 200
❌ THẤT BẠI: Server trả về JSON hợp lệ nhưng Data là NULL.
   -> Đây chính là nguyên nhân lỗi 'NoneType is not subscriptable'

----- 🧪 TEST CASE: Chỉ truyền Năm (EndDate = '2025') -----
🔗 URL Thực tế: https://cafef.vn/du-lieu/Ajax/PageNew/FinanceReport.ashx?Symbol=HRC&Type=1&ReportType=QUY&TotalRow=100&EndDate=2025
📡 Status Code: 200
❌ THẤT BẠI: Server trả về JSON hợp lệ nhưng Data là NULL.
   -> Đây chính là nguyên nhân lỗi 'NoneType is not subscriptable'

----- 🧪 TEST CASE: Truyền Tháng-Năm (EndDate = '12-2025') -----
🔗 URL Thực tế: https://cafef.vn/du-lieu/Ajax/PageNew/FinanceReport.ashx?Symbol=HRC&Type=1&ReportType=QUY&TotalRow=100&EndDate=12-2025
📡 Status Code: 200
✅ THÀNH CÔNG! Tìm thấy 86 bản ghi.
   Dữ liệu mới nhất: Quý 4/2005
